# 5주차 3차시: 재순위화 · 필터링 · 압축 · 평가

| 단계 | 기법 |
|---|---|
| 재순위화 (Reranking) | BM25 · LLM Pairwise · Hybrid · Listwise |
| 필터링 (Filtering) | Fixed / Dynamic / Score Gap · Adaptive |
| 문서 압축 (Compression) | LLM 추출 · MMR |
| 평가 (Evaluation) | AP · mAP · ILS · A/B Test · LLM-as-Judge |

In [1]:
import os
import re
import time
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import ContextualCompressionRetriever, EnsembleRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from dotenv import load_dotenv

load_dotenv()

MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL)
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")


In [2]:
documents = [
    Document(page_content="트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.", metadata={"id": "d1"}),
    Document(page_content="BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.", metadata={"id": "d2"}),
    Document(page_content="GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.", metadata={"id": "d3"}),
    Document(page_content="RAG는 검색 증강 생성 기법으로 외부 지식을 LLM에 결합하여 할루시네이션을 줄입니다.", metadata={"id": "d4"}),
    Document(page_content="벡터 데이터베이스는 임베딩 벡터를 저장하고 유사도 기반 검색을 수행합니다. FAISS, Pinecone 등이 있습니다.", metadata={"id": "d5"}),
    Document(page_content="파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.", metadata={"id": "d6"}),
    Document(page_content="프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.", metadata={"id": "d7"}),
    Document(page_content="토큰화는 텍스트를 모델이 처리할 수 있는 단위로 분할하는 과정입니다. BPE, WordPiece 등이 사용됩니다.", metadata={"id": "d8"}),
]


In [3]:
vectorstore = FAISS.from_documents(documents, embeddings_model)
bm25_retriever = BM25Retriever.from_documents(documents, k=5)

In [4]:
doc_embeddings = {}

def get_embedding(text):
    return np.array(embeddings_model.embed_query(text))

for doc in doc_embeddings:
    doc_embeddings[doc.metadata['id']] = get_embedding(doc.page_content)

## 검색 결과 재순위화 (Reranking)

기본 벡터 검색은 임베딩 유사도만 사용 → 정밀도에 한계가 있음.  
재순위화는 초기 검색 결과를 더 정교한 기준으로 다시 정렬하는 과정.

### Bi-encoder vs Cross-encoder

| 방식 | 구조 | 속도 | 정밀도 | 활용 |
|---|---|---|---|---|
| Bi-encoder | query, doc 각각 인코딩 → 코사인 유사도 | 빠름 | 보통 | 초기 검색 (Recall) |
| Cross-encoder | query + doc 합쳐서 인코딩 | 느림 | 높음 | 재순위화 (Precision) |

### 파이프라인

```
Query → [Bi-encoder 벡터 검색] → 후보 Top-K → [Reranking] → 정제 Top-N → [LLM] → 답변
```

### 기본 벡터 검색 및 키워드 재순위화

- `vector_search`: FAISS 유사도 검색으로 Top-K 반환
- `keyword_rerank`: 쿼리 키워드 매칭 수 기준 재정렬. 특정 키워드가 중요한 도메인에서 효과적
- `rank_change`: 재순위화 전후 순위 변동을 DataFrame으로 비교

In [9]:
def vector_search(query, vectorstore, top_k=5):
    results = vectorstore.similarity_search_with_score(query, k=top_k)
    return [(doc, 1.0 / (1.0 + score)) for doc, score in results]

query = "트랜스포머와 BERT의 관계"
results = vector_search(query, vectorstore)
results

[(Document(id='f494f542-f2b0-4227-845a-0046521cccb0', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  np.float32(0.5248781)),
 (Document(id='607ff184-e129-486c-88ba-b27647c062de', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  np.float32(0.4639076)),
 (Document(id='e77d3bbb-b96d-4dac-ad1c-900a91197c25', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  np.float32(0.41887715)),
 (Document(id='501c24ec-a920-4ad7-9375-2986c92c7c6a', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  np.float32(0.4127892)),
 (Document(id='a1b6cd71-3456-4fd6-8be9-c88dbdca066b', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  np.float32(0.39725193))]

In [10]:
def keyword_rerank(query, search_results):
    query_terms = set(query.lower().split())
    reranked = []
    
    for doc, orig_score in search_results:
        doc_terms = doc.page_content.lower().split()
        keyword_hits = sum(1 for t in doc_terms if t in query_terms)   # [1, 1]
        new_score = orig_score + 0.1 * keyword_hits
        reranked.append((doc, new_score, orig_score))
        
    reranked.sort(key = lambda x: x[1], reverse=True)
    return reranked

In [11]:
reranked = keyword_rerank(query, results)
reranked

[(Document(id='f494f542-f2b0-4227-845a-0046521cccb0', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  np.float32(0.5248781),
  np.float32(0.5248781)),
 (Document(id='607ff184-e129-486c-88ba-b27647c062de', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  np.float32(0.4639076),
  np.float32(0.4639076)),
 (Document(id='e77d3bbb-b96d-4dac-ad1c-900a91197c25', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  np.float32(0.41887715),
  np.float32(0.41887715)),
 (Document(id='501c24ec-a920-4ad7-9375-2986c92c7c6a', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  np.float32(0.4127892),
  np.float32(0.4127892)),
 (Document(id='a1b6cd71-3456-4fd6-8be9-c88dbdca066b', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  np.float32(0.39725193),
  np.float32(0.39725193)

In [12]:
def rank_change(original_results, reranked_results):
    
    orig_ranks = {doc_.metadata['id']: i+1 for i, (doc_, _) in enumerate(original_results)}
    new_ranks = {doc_.metadata['id']: i+1 for i, (doc_, _, _) in enumerate(reranked_results)}
    
    changes = []
    for doc_id in orig_ranks:
        old_r = orig_ranks[doc_id]
        new_r = new_ranks.get(doc_id, -1)
        
        changes.append({
            'doc_id' : doc_id,
            'before' : old_r,
            'after' : new_r,
            'change' : old_r - new_r
        })
        
    return pd.DataFrame(changes).sort_values('after')


In [13]:
df = rank_change(results, reranked)
df

,doc_id,before,after,change
0,d1,1,1,0
1,d2,2,2,0
2,d3,3,3,0
3,d6,4,4,0
4,d7,5,5,0


### BM25 기반 재순위화

TF-IDF를 개선한 키워드 기반 랭킹 알고리즘.

- `k1` (기본 1.5): 단어 빈도 포화 계수. 높을수록 빈도 영향 증가
- `b` (기본 0.75): 문서 길이 정규화 계수. 1에 가까울수록 길이 영향 증가

In [17]:
# TF-IDF 
class BM25Reranker: 
    def __init__(self, k1 = 1.5, b=0.75):
        self.k1 = k1
        self.b = b
    
    def rerank(self, query, search_results):
        docs = [doc for doc, _ in search_results]
        tokenized = [doc.page_content.lower().split() for doc in docs]
        
        avg_dl = np.mean([len(t) for t in tokenized])
        
        df_count = Counter()
        for tokens in tokenized:
            for t in set(tokens):
                df_count[t] += 1
        N = len(docs)
        
        query_tokens = query.lower().split()
        scored = []
        
        for i, doc in enumerate(docs):
            score = 0.0
            doc_len = len(tokenized[i])
            tf_count = Counter(tokenized[i])
            
            for qt in query_tokens :
                tf = tf_count.get(qt, 0)
                if tf == 0:
                    continue
                
                df = df_count.get(qt, 0)
                idf = math.log((N-df + 0.5) / (df + 0.5) + 1)    # tf-idf : tf x idf   10 -> log(10) : 1 (0.1 -> (-1))
                numerator = tf * (self.k1 + 1)
                denominator = tf + self.k1 * (1 - self.b + self.b * doc_len /avg_dl)
                score +=  idf * numerator / denominator
        
            scored.append((doc, score))
        
        scored.sort(key = lambda x: x[1], reverse=True)
        return scored
                
        
        

In [18]:
bm25_reranker = BM25Reranker()
bm25_results = bm25_reranker.rerank(query, results)

In [19]:
bm25_results

[(Document(id='f494f542-f2b0-4227-845a-0046521cccb0', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  0.0),
 (Document(id='607ff184-e129-486c-88ba-b27647c062de', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  0.0),
 (Document(id='e77d3bbb-b96d-4dac-ad1c-900a91197c25', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  0.0),
 (Document(id='501c24ec-a920-4ad7-9375-2986c92c7c6a', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  0.0),
 (Document(id='a1b6cd71-3456-4fd6-8be9-c88dbdca066b', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  0.0)]

### LLM 기반 재순위화 (Pairwise Reranking)

LLM에게 문서 목록을 보여주고 관련성 순 재정렬 요청.

- 장점: 문서 간 의미적 관계를 LLM이 이해 → 정교한 순위 결정
- 단점: API 비용 발생, 문서 수에 비례해 느려짐

In [21]:
def llm_rerank(query, search_results, top_k=3):
    docs_text = '\n'.join(
        f"[{doc.metadata['id']}] {doc.page_content}" for doc, _ in search_results
    )
    
    score_template = ", ".join(f'"{doc.metadata["id"]}": 0.0-1.0' for doc, _ in search_results)
    
    scoring_chain = ChatPromptTemplate.from_messages([
        ('system', '당신은 scoring 시스템 입니다. 항상 json 형태로 출력하세요'),
        ('human', """다음 쿼리에 대해 각 문서의 관련성을 0.0~1.0으로 평가하세요.
        
        쿼리 : {query}
        
        문서들 :
        {docs_text}
        
        "스트링으로 묶지말고" JSON으로 답하세요:
        {{"scores" : {{{score_template}}}}}""")
    ]) | llm | StrOutputParser()
    
    result = scoring_chain.invoke({
        'query' : query,
        'docs_text' : docs_text,
        'score_template' : score_template
    })
    
    cleaned = result.strip()
    if cleaned.startswith('```json'):
        cleaned = cleaned.replace('```json', '')
        cleaned = cleaned.replace('```', '')
        
    parsed = json.loads(cleaned)
    scores = parsed.get('scores', {})
    
    scored = []
    
    for doc, orig in search_results:
        rerank_score = scores.get(doc.metadata['id'],0.0)
        scored.append((doc, float(rerank_score), orig))
        
    scored.sort(key = lambda x : x[1], reverse=True)
    return scored[:top_k]

In [24]:
llm_results = llm_rerank(query, results, top_k=3)

In [25]:
llm_results

[(Document(id='607ff184-e129-486c-88ba-b27647c062de', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  1.0,
  np.float32(0.4639076)),
 (Document(id='e77d3bbb-b96d-4dac-ad1c-900a91197c25', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  0.4,
  np.float32(0.41887715)),
 (Document(id='f494f542-f2b0-4227-845a-0046521cccb0', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  0.3,
  np.float32(0.5248781))]

### 하이브리드 재순위화

벡터 검색(의미 기반) + BM25(키워드 기반) 점수를 가중 결합.

```
최종 점수 = (1 - bm25_weight) × 벡터 점수 + bm25_weight × BM25 점수
```

두 방식의 장점을 결합해 Recall과 Precision 균형을 높임.

In [26]:
def hybrid_rerank(query, search_results, bm25_weight = 0.3):
    bm25 = BM25Reranker()
    bm25_scored = bm25.rerank(query, search_results)
    bm25_map = {doc.metadata['id'] : score for doc, score in bm25_scored}
    
    llm_scored = llm_rerank(query, search_results, top_k = len(search_results))
    llm_map = {doc.metadata['id'] : score for doc, score, _ in llm_scored}
    
    def normalize(scores_dict):
        vals = list(scores_dict.values())
        min_, max_ = min(vals), max(vals)
        range_ = max_ - min_ if max_ > min_ else 1e-8
        return {k : (v - min_) / range_ for k, v in scores_dict.items()}
    
    bm25_norm = normalize(bm25_map)
    llm_norm = normalize(llm_map)
    
    combined = []
    for doc, _ in search_results:
        did = doc.metadata['id']
        score = bm25_weight * bm25_norm.get(did, 0) + (1- bm25_weight) * llm_norm.get(did, 0)
        
        combined.append((doc, score))
    combined.sort(key = lambda x:x[1], reverse=True)
    return combined

In [27]:
hybrid = hybrid_rerank(query, results)

In [28]:
hybrid

[(Document(id='607ff184-e129-486c-88ba-b27647c062de', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  0.7),
 (Document(id='e77d3bbb-b96d-4dac-ad1c-900a91197c25', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  0.35),
 (Document(id='f494f542-f2b0-4227-845a-0046521cccb0', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  0.2625),
 (Document(id='501c24ec-a920-4ad7-9375-2986c92c7c6a', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  0.0875),
 (Document(id='a1b6cd71-3456-4fd6-8be9-c88dbdca066b', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  0.0)]

### LLM Listwise 재순위화

| 방식 | 비교 대상 | 특징 |
|---|---|---|
| Pointwise | 문서 1개씩 독립 평가 | 빠르지만 문서 간 관계 무시 |
| Listwise | 문서 목록 전체를 한 번에 평가 | 상대 순위 파악 가능, API 호출 적음 |

N개 문서를 한꺼번에 제시하고 순위 번호를 직접 출력하게 하는 방식.

In [32]:
def llm_listwise_rerank(query, search_results):
    docs_text = '\n'.join(
            f"{i+1}. [{doc.metadata['id']}] {doc.page_content[:80]}" for i, (doc, _) in enumerate(search_results)
    )
    
    ranking_chain = ChatPromptTemplate.from_messages([
        ('system', "당신은 문서 랭킹 시스템입니다"),
        ('human', """다음 문서들을 쿼리와의 관련성 순서로 정렬하세요.
        
        쿼리 : {query}
        
        문서들 : {docs_text}
        
        가장 관련성 높은 순서대로 문서 번호를 쉼표로 나열하세요 (예: 3,1,5,2,4):""")
    ]) | llm | StrOutputParser()
    
    result = ranking_chain.invoke({'query' : query, 'docs_text' : docs_text})
    
    order = [int(x.strip()) for x in result.strip().split(',')]
    
    docs_list = [doc for doc, _ in search_results]
    reranked = []
    for rank, idx in enumerate(order):
        if 1<=idx<=len(docs_list):
            docs = docs_list[idx-1]
            reranked.append((docs, 1.0-rank*0.1))
            
    return reranked

In [33]:
listwise = llm_listwise_rerank(query, results)

In [34]:
listwise

[(Document(id='607ff184-e129-486c-88ba-b27647c062de', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  1.0),
 (Document(id='f494f542-f2b0-4227-845a-0046521cccb0', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  0.9),
 (Document(id='e77d3bbb-b96d-4dac-ad1c-900a91197c25', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  0.8),
 (Document(id='501c24ec-a920-4ad7-9375-2986c92c7c6a', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  0.7),
 (Document(id='a1b6cd71-3456-4fd6-8be9-c88dbdca066b', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  0.6)]

## 스코어 필터링 (Score Filtering)

재순위화 이후 관련성이 낮은 문서를 제거해 컨텍스트 품질을 높임.  
LLM 컨텍스트 윈도우를 효율적으로 활용하기 위해 필요.

| 전략 | 방식 | 특징 |
|---|---|---|
| Fixed Threshold | 고정 점수 이상만 유지 | 단순, 도메인별 임계값 설정 필요 |
| Dynamic Threshold | 평균 - 표준편차 기준 | 데이터 분포에 적응적 |
| Score Gap | 점수 급락 구간에서 절단 | 자연스러운 경계 탐지 |

In [35]:
class ScoreFilter:
    
    @staticmethod
    def fixed_threshold(scored_docs, threshold= 0.5):
        return [(doc, s) for doc, s in scored_docs if s >= threshold]
    
    @staticmethod
    def dynamic_threshold(scored_docs, std_factor) : 
        scores = [s for _, s in scored_docs]
        threshold = np.mean(scores) - std_factor * np.std(scores)
        return [(doc, s) for doc, s in scored_docs if s >= threshold]
    
    @staticmethod
    def score_gap(scored_docs, min_docs=2):
        if len(scored_docs) <= min_docs:
            return scored_docs
        
        scores = [s for _, s in scored_docs]
        gaps = [(scores[i] - scores[i+1], i+1) for i in range(len(scores)-1)]
        max_gap_idx = max(gaps, key=lambda x:x[0])[1]
        cut = max(min_docs, max_gap_idx)
        return scored_docs[:cut]



In [37]:
test_scores = [(Document(page_content="", metadata = {'id': f'd{i}'}), s) for i, s in enumerate([0.9, 0.85, 0.7, 0.5, 0.2])]
test_scores

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5),
 (Document(metadata={'id': 'd4'}, page_content=''), 0.2)]

#### Fixed Threshold

점수가 threshold(0.5) 이상인 문서만 반환.

In [38]:
filtered = ScoreFilter.fixed_threshold(test_scores, threshold= 0.5)

In [39]:
filtered

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5)]

#### Dynamic Threshold

`평균 - std_factor × 표준편차` 이상인 문서만 반환. 점수 분포에 따라 임계값 자동 조정.

In [40]:
filtered = ScoreFilter.dynamic_threshold(test_scores, std_factor=1)
filtered

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5)]

#### Score Gap

연속된 문서 간 점수 차이가 가장 큰 지점에서 절단. 자연스러운 품질 경계 탐지에 유용.

In [41]:
filtered = ScoreFilter.score_gap(test_scores, min_docs=2)
filtered

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5)]

### 적응형 필터링 (AdaptiveFilter)

문서 수 · 점수 분포를 분석해 Fixed / Dynamic / Score Gap 중 가장 적합한 전략을 자동 선택.

In [42]:
class AdaptiveFilter:
    
    # 점수의 분포에 따라서 score filtering 전략을 자동으로 선택
    
    @staticmethod # decorator
    def adapt_filter(scored_docs):
        scores = [s for _, s in scored_docs]
        std = np.std(scores)
        score_range = max(scores) - min(scores)
        
        if std > 0.2:
            result = ScoreFilter.score_gap(test_scores, min_docs=2)
        elif score_range < 0.1:
            result = scored_docs[:len(scored_docs)//2]
        else:
            result = ScoreFilter.dynamic_threshold(test_scores, std_factor=1)
        
        return result
    

In [43]:
filtered = AdaptiveFilter.adapt_filter(test_scores)
filtered

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5)]

## 문서 압축 (Context Compression)

검색 문서 전체를 LLM에 넘기면 토큰 비용 증가 + 노이즈 증가.  
질문 관련 핵심 내용만 추출해 컨텍스트 품질을 높이는 과정.

| 방식 | 설명 | 특징 |
|---|---|---|
| LLM 기반 압축 | LLM이 관련 문장만 추출 | 높은 정확도, API 비용 발생 |
| MMR 기반 압축 | 관련성 + 다양성 균형 최적화 | 빠름, 중복 제거 효과적 |

In [54]:
long_doc = "트랜스포머는 2017년 구글이 발표한 아키텍처입니다. Self-Attention 메커니즘이 핵심입니다. 이전의 RNN, LSTM과 달리 병렬 처리가 가능합니다. BERT와 GPT 모두 트랜스포머를 기반으로 합니다. 자연어 처리뿐 아니라 컴퓨터 비전에서도 활용됩니다"
query = "트랜스포머와 BERT의 관계"
compressed = extractive_compress(query, long_doc, max_sentences=3)
compressed

'트랜스포머는 2017년 구글이 발표한 아키텍처입니다. Self-Attention 메커니즘이 핵심입니다. 이전의 RNN, LSTM과 달리 병렬 처리가 가능합니다.'

### LLM 기반 압축

압축 프롬프트를 정의하고 질문 관련 문장만 추출하도록 LLM에 요청.

In [55]:
compress_chain = ChatPromptTemplate.from_messages([
    ("system", "당신은 문서 요약 전문가입니다"),
    ("human", """다음 문서에서 쿼리에 답하는데 필요한 핵심 정보만 추출하세요.
    불필요한 내용은 제거하고, {max_tokens}자 이내로 압축하세요.
    
    쿼리 : {query}
    문서 : {document}
    
    압축결과:""")
]) | llm | StrOutputParser()

In [56]:
def llm_compress(query, document, max_tokens=100):
    return compress_chain.invoke({
        'query' : query,
        'document' : document,
        'max_tokens' : max_tokens
    })

In [57]:
compressed_llm = llm_compress(query, long_doc)

In [58]:
compressed_llm

'트랜스포머는 2017년 발표된 아키텍처로, BERT와 GPT의 기반이 됩니다. Self-Attention 메커니즘이 핵심입니다.'

### Contextual Compression Retriever

`LLMChainExtractor` + `ContextualCompressionRetriever`를 결합해 검색과 압축을 하나의 파이프라인으로 처리.

In [59]:
compressor = LLMChainExtractor.from_llm(llm)
compressor_retriever = ContextualCompressionRetriever(base_compressor = compressor, 
                                                      base_retriever=vectorstore.as_retriever(search_kwargs={'k':3}))

In [60]:
compressed_docs = compressor_retriever.invoke(query)

In [61]:
compressed_docs

[Document(metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
 Document(metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.')]

### 압축 품질 평가

- **압축률**: 원본 대비 압축 후 길이 비율 (낮을수록 많이 압축)
- **키워드 보존율**: 쿼리 핵심 단어가 얼마나 남았는지
- **의미 유사도**: 원본과 압축본의 임베딩 코사인 유사도

In [62]:
# 압축 품질 평가 : 압축률(200자 -> 100자), 키워드 보존율, 의미 보존도 (emb)
def evaluate_compression(original, compressed, query):
    # 1) 압축율
    ratio = len(compressed)  / len(original)
    
    # 2) 키워드 보존율
    query_terms = set(query.lower().split())
    orig_terms = set(original.lower().split())
    comp_terms = set(compressed.lower().split())
    
    orig_query_terms = query_terms & orig_terms
    comp_query_terms = query_terms & comp_terms
    keyword_coverage = len(comp_query_terms) / len(orig_query_terms) if orig_query_terms else 1.0
    
    # 3) 의미 보존도
    original_emb = np.array(embeddings_model.embed_query(original))  # get_embedding(original)
    compressed_emb = get_embedding(compressed)
    
    semantic_similarity = np.dot(original_emb, compressed_emb) / (np.linalg.norm(original_emb) * np.linalg.norm(compressed_emb))
    
    return {
        'compression_ratio' : ratio,
        'keyword_coverage' : keyword_coverage,
        'semantic_similarity' : semantic_similarity
    }

In [63]:
evaluate_compression(long_doc, compressed_llm, query)

{'compression_ratio': 0.48299319727891155,
 'keyword_coverage': 1.0,
 'semantic_similarity': np.float64(0.8632305328608311)}

### MMR 기반 압축 (Maximum Marginal Relevance)

관련성(Relevance)과 다양성(Diversity)을 동시에 최적화.

```
MMR(s) = λ × Relevance(s, query) - (1-λ) × max Similarity(s, selected)
```

- λ 높음 → 관련성 우선 (정보량 집중)
- λ 낮음 → 다양성 우선 (중복 제거)

In [65]:
def cosine_sim(original_emb, compressed_emb):
    return np.dot(original_emb, compressed_emb) / (np.linalg.norm(original_emb) * np.linalg.norm(compressed_emb))

In [66]:
def mmr_compress(query, document, max_sentences=3, param=0.5):
    sentences = re.split(r'[.!?]\s*', document)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    
    if len(sentences) <= max_sentences:
        return ". ".join(sentences) + "."
    
    query_emb = get_embedding(query)
    sent_embs = [get_embedding(s) for s in sentences]
    
    selected = []
    remaining = list(range(len(sentences)))
    
    for _ in range(max_sentences):
        best_score = -float('inf')
        best_idx = -1
        
        for idx in remaining:
            relevance = cosine_sim(query_emb, sent_embs[idx])
            
            if selected:
                max_sim = max(cosine_sim(sent_embs[idx], sent_embs[s]) for s in selected)
            else:
                max_sim = 0.0
                
            mmr = param * relevance - (1 - param) * max_sim
            
            if mmr > best_score:
                best_score = mmr
                best_idx = idx
        
        selected.append(best_idx)
        remaining.remove(best_idx)
        
        
    return ". ".join(sentences[i] for i in sorted(selected)) + "."

#### Relevance 중심 (param=0.7)

관련성 70% 반영. 쿼리와 직접 관련된 문장 위주로 선택.

In [67]:
mmr_compress(query, long_doc, max_sentences=2, param=0.7)

'트랜스포머는 2017년 구글이 발표한 아키텍처입니다. BERT와 GPT 모두 트랜스포머를 기반으로 합니다.'

#### Diversity 중심 (param=0.3)

다양성 70% 반영. 중복 없이 다양한 관점의 문장 선택.

In [68]:
mmr_compress(query, long_doc, max_sentences=2, param=0.3)

'트랜스포머는 2017년 구글이 발표한 아키텍처입니다. 이전의 RNN, LSTM과 달리 병렬 처리가 가능합니다.'

## 검색 품질 평가 지표

재순위화 · 필터링이 실제로 검색 품질을 개선했는지 정량 측정.

### AP (Average Precision)

각 관련 문서 위치에서의 Precision 평균. 1에 가까울수록 관련 문서가 상위에 밀집.

```
AP = (1/|R|) × Σ P@k × 1[d_k ∈ R]
```

In [69]:
def average_precision(retrieved_ids, relevant_ids):
    relevant_set = set(relevant_ids)
    hits = 0
    sum_precision = 0.0
    
    for i, doc_id in enumerate(retrieved_ids):
        if doc_id in relevant_set:
            hits += 1
            precision_at_i = hits / (i+1)
            sum_precision += precision_at_i
            
    return sum_precision/len(relevant_set) if relevant_set else 0.0



In [72]:
relevant = {'d1', 'd2', 'd3'}
orig_order = [doc.metadata['id'] for doc, _ in results]
orig_order

['d1', 'd2', 'd3', 'd6', 'd7']

In [73]:
ap = average_precision(orig_order, relevant)

In [74]:
ap

1.0

### mAP (Mean Average Precision)

여러 질문에 대한 AP의 평균. 시스템 전체 성능을 하나의 수치로 표현.

In [70]:
def mean_average_precision(query_results):
    # query_results : {query1 : (retrieved_ids, relvant_ids), query2 : (retrieved_ids, relvant_ids)}
    
    aps = []
    for query, (retrieved, relevant) in query_results.items():
        ap = average_precision(retrieved, relevant)
        aps.append(ap)
        
    # aps = [0.7556, 0.7777, 0.3333, ,,,,]
    map_score = np.mean(aps)
    return map_score

In [71]:
results

[(Document(id='f494f542-f2b0-4227-845a-0046521cccb0', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  np.float32(0.5248781)),
 (Document(id='607ff184-e129-486c-88ba-b27647c062de', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  np.float32(0.4639076)),
 (Document(id='e77d3bbb-b96d-4dac-ad1c-900a91197c25', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  np.float32(0.41887715)),
 (Document(id='501c24ec-a920-4ad7-9375-2986c92c7c6a', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  np.float32(0.4127892)),
 (Document(id='a1b6cd71-3456-4fd6-8be9-c88dbdca066b', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  np.float32(0.39725193))]

### ILS (Intra-List Similarity)

검색 결과 목록 내 문서들의 상호 유사도.

- ILS 낮음 → 결과가 다양 (사용자에게 다양한 정보 제공)
- ILS 높음 → 결과가 유사 (중복 위험)

In [75]:
def intra_list_similarity(doc_ids, doc_embeddings, top_k=5):
    ids = doc_ids[:top_k]
    embs = [doc_embeddings[did] for did in ids if did in doc_embeddings]
    
    if len(embs) < 2:
        return 0.0
    
    sims = []
    for i in range(len(embs)):
        for j in range(i+1, len(embs)):
            sims.append(cosine_sim(embs[i], embs[j]))
            
    return np.mean(sims)

In [76]:
ils_score = intra_list_similarity(orig_order, doc_embeddings)
ils_score

0.0

## A/B Test

A기존/ B변경후

A안/ B안 등등

A 디자인 : 조회수 100, 결제 200

B 디자인 :        120       190

t-test : p-value 와 유의수준(alpha) 의 관계를 보고, 어떻게 될지를 알려줌

통계 검증을 할때는 귀무가설 <-> 대립가설
- 귀무가설: A, B가 차이가 없다. (Null hypothesis)
- 대립가설: A, B가 차이가 있다.

파이썬 함수인 t-test를 쓰면 -> p-value가 나옴
- p-value의 의미? 차이가 없으나, 그러나 지금의 결과가 나올 확률

p-value : 0.01 -> A, B가 차이가 없는데, 지금의 결과가 나올 확률이 1%라는 의미 -> 귀무가설 기각 가능.
- 의미? 귀무가설이 참일 때, 현재 관측된 결과(또는 보다 극단적인 수치)가 나올 확률

mAP, p-value

t-test 가정: A, B 두 집단의 분포가 비슷하다 (분산이 일치한다)
- 분산이 다르다 -> Welch's t-test

모집단이 같은 경우 -> 우리반의 수학점수, 우리반의 영어점수
모집단이 다른 경우 : 우리반의 수학점수 <-> 옆반의 수학점수

cohen's d -> 두 집단의 차이가 얼만큼 있나 (큰가 작은가)
- d = (mean_a - mean_b) / s(표준편차) 로 나눠줌. 집단별 표준편차가 다른데 어떤걸 쓸건지 (s: a,b 공통의 표준편차: pooled std)

### 구현

- `ab_test()`: 개선 전(A)과 개선 후(B)의 MAP 점수 목록에 t-검정 수행
- `p_value < alpha(0.05)` → 귀무가설 기각 → 유의미한 차이 존재
- `cohen's d > 0.5` → 중간 이상의 효과 크기

In [85]:
# scores_a : [0.8666666666666667, 1.0, 0.5833333333333333]
def ab_test(scores_a, scores_b, alpha=0.05):
    t_stat, p_val = stats.ttest_rel(scores_a, scores_b)
    
    mean_a = np.mean(scores_a)
    mean_b = np.mean(scores_b)
    
    pooled_std = np.sqrt((np.std(scores_a)**2 + np.std(scores_b)**2)/2)
    cohens_d = (mean_a - mean_b) / pooled_std if pooled_std > 0 else 0.0
    
    return {
        'mean_A' : mean_a,
        'mean_B' : mean_b,
        'p_val' : p_val,
        'cohens_d' : cohens_d,
        'significant' : p_val < alpha
    }
    

In [86]:
test_queries = {
    '트랜스포머 아키텍처' : ({'d1', 'd2', 'd3'}),
    '벡터 검색 방법' : ({'d5'}),
    'LLM 학습 방법' : ({'d6', 'd7'})
}

aps_before = []
aps_after = []
bm25_reranker = BM25Reranker()
for q, relevant in test_queries.items():
    orig = vector_search(q, vectorstore)
    reranked = bm25_reranker.rerank(q, orig)
    
    orig_ids = [doc.metadata['id'] for doc, _ in orig]
    reranked_ids = [doc.metadata['id'] for doc, _ in reranked]
    
    aps_before.append(average_precision(orig_ids, relevant))
    aps_after.append(average_precision(reranked_ids, relevant))
    

In [87]:
aps_before

[0.8666666666666667, 1.0, 0.5833333333333333]

In [88]:
result = ab_test(aps_before, aps_after, alpha=0.05)

In [89]:
result

{'mean_A': np.float64(0.8166666666666668),
 'mean_B': np.float64(0.6944444444444443),
 'p_val': np.float64(0.590905598181919),
 'cohens_d': np.float64(0.6187983455093142),
 'significant': np.False_}

## LLM-as-Judge 평가

### 기존 지표의 한계

| 지표 | 방식 | 한계 |
|---|---|---|
| BLEU | n-gram 겹침 (정밀도 중심) | 의미적 동의어 무시, 창의적 표현 불이익 |
| ROUGE | n-gram 겹침 (재현율 중심) | 동일한 한계, 어순 무관 |

### LLM-as-Judge 방식

| 방식 | 입력 | 출력 | 특징 |
|---|---|---|---|
| Pointwise | 질문 + 답변 1개 | 점수 (1~5) | 절대 평가, 빠름 |
| Pairwise | 질문 + 답변 2개 | A / B / Tie | 상대 비교, 더 정밀 |
| Reference-based | 질문 + 정답 + 답변 | 점수 | 정답 기준 평가 |

LLM은 의미론적 이해 기반 평가 → 동의어 · 패러프레이징에 강건.

### BLEU / ROUGE

`simple_rouge_f1`: 정밀도와 재현율의 조화평균 기반 간이 ROUGE 구현.

In [96]:
def simple_rouge_f1(reference, candidate):
    ref_words = Counter(reference.split())
    cand_words = Counter(candidate.split())
    overlap = sum(min(ref_words[w], cand_words.get(w, 0)) for w in ref_words)
    r = overlap / sum(ref_words.values()) if ref_words else 0
    p = overlap / sum(cand_words.values()) if cand_words else 0
    return 2*p*r / (p+r) if (p+r) >0 else 0


In [97]:
question = "딥러닝이란?"
reference = "딥러닝은 다층 신경망을 사용하여 데이터에서 복잡한 패턴을 학습하는 머신러닝 방법입니다"

test_answers = [
    ("딥러닝은 다층 신경망을 사용하여 데이터에서 복잡한 패턴을 학습하는 머신러닝 방법입니다", "정답 복사"),
    ("인공 신경망의 층을 깊게 쌓아 복잡한 문제를 풀 수 있는 AI 기술이에요", "좋은 패러프레이즈"),
    ("딥러닝 딥러닝 신경망 데이터 학습 패턴 머신러닝", "키워드 나열"),
]


In [108]:
rows = []
for ans, label in test_answers:
    rouge = simple_rouge_f1(reference, ans)
    llm_result = basic_judge2(question, ans)
    
    print(llm_result, type(llm_result))
    
    rows.append({
        '유형' : label,
        'rouge' : {rouge},
        'llm_score' : {llm_result.get('score', 0)},
        'llm_confidence' : {llm_result.get('confidence', 0)}
    })

{'score': 4, 'confidence': 'mid'} <class 'dict'>
{'score': 4, 'confidence': 'mid'} <class 'dict'>
{'score': 2, 'confidence': 'low'} <class 'dict'>


In [109]:
rows

[{'유형': '정답 복사', 'rouge': {1.0}, 'llm_score': {4}, 'llm_confidence': {'mid'}},
 {'유형': '좋은 패러프레이즈',
  'rouge': {0.0909090909090909},
  'llm_score': {4},
  'llm_confidence': {'mid'}},
 {'유형': '키워드 나열',
  'rouge': {0.11764705882352941},
  'llm_score': {2},
  'llm_confidence': {'low'}}]

### Pointwise 평가

루브릭(Rubric) 기반으로 답변 1개를 1~5점으로 평가.

In [128]:
RUBRIC = {
    5: "정확하고 완전하며, 예시와 설명이 풍부하다",
    4: "정확하고 핵심을 다루지만, 일부 세부사항이 부족하다",
    3: "대체로 정확하지만, 중요한 내용 일부가 누락되었다",
    2: "부분적으로만 정확하거나, 핵심을 빗나갔다",
    1: "부정확하거나 질문과 무관하다",
}

def pointwise_judge(question, answer, rubric = RUBRIC):
    rubric_text = '\n'.join(
        f' {k} 점: {v}' for k, v in sorted(rubric.items(), reverse=True)
    )
    
    prompt = f"""다음 답변을 아래 루브릭에 따라 평가해주세요.
    
    질문 : {question}
    답변 : {answer}
    
    루브릭 :
    {rubric_text}
    
    반드시 JSON으로 답하세요:
    {{"score": 1-5, "reasoning" : "이유"}}"""
    
    result = llm.invoke(prompt).content
#     print(result)
    result = result.replace('```json', '').replace('```', '')
    return json.loads(result)

In [90]:
def basic_judge(question, answer, scale='1~5'):
    """기본 LLM평가: 질문-답변 쌍을 점수화"""
    prompt = f"""다음 답변을 {scale}스케일로 평가하세요.
    
    질문 : {question}
    답변 : {answer}
    
    평가 기준
    - 정확성 : 사실적으로 올바른가?
    - 완전성 : 질문에 충분히 답했는가?
    - 명확성 : 이해하기 쉽게 설명했는가?
    
    반드시 아래 형식으로 답하세요:
    점수 : [숫자]
    이유 : [한 줄 설명]"""
    
    result = llm.invoke(prompt).content
    return result

In [91]:
question = "파이썬의 장점은 무엇인가요?"
answers = [
    "파이썬은 간결한 문법으로 초보자도 쉽게 배울 수 있으며, 풍부한 라이브러리와 활발한 커뮤니티가 있습니다.",
    "파이썬 좋아요.",
    "자바스크립트는 웹 개발에 주로 사용됩니다.",
]


In [92]:
for i, ans in enumerate(answers):
    result = basic_judge(question, ans)
    print(f"평가 : {result}")

평가 : 점수 : 5  
이유 : 답변이 파이썬의 주요 장점인 간결한 문법, 풍부한 라이브러리, 활발한 커뮤니티를 정확하고 명확하게 설명하였으며, 질문에 충분히 답변하였습니다.
평가 : 점수 : 1  
이유 : 답변이 너무 간단하여 질문에 대한 정확하고 완전한 정보를 제공하지 못하고 있습니다.
평가 : 점수 : 1  
이유 : 답변이 질문과 관련이 없으며 파이썬의 장점을 전혀 설명하지 않고 있다.


#### Pointwise + ROUGE 결합

ROUGE F1과 LLM 점수를 함께 비교. 어휘 겹침과 의미 이해 기반 평가의 차이 확인 가능.

In [104]:
def basic_judge2(question, answer, scale='1~5'):
    """기본 LLM평가: 질문-답변 쌍을 점수화"""
    prompt = f"""다음 답변을 {scale}스케일로 평가하세요. 답변은 반드시 JSON으로 반환하세요
    
    질문 : {question}
    답변 : {answer}
    
    평가 기준
    - 정확성 : 사실적으로 올바른가?
    - 완전성 : 질문에 충분히 답했는가?
    - 명확성 : 이해하기 쉽게 설명했는가?
    
    반드시 아래 형식으로 답하세요:
    {{"score": 1~5점수, 'confidence' : 'high/mid/low'}}"""
    
    result = llm.invoke(prompt).content
    
    cleaned = result.strip()
    
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```")[1]
        if cleaned.startswith("json"):
            cleaned = cleaned[4:]
        return json.loads(cleaned)
#     return result

In [129]:
question = "REST API와 GraphQL의 차이점을 설명해주세요"
answers = [
    "REST는 리소스 단위 URL에 HTTP 메서드를 사용하고, GraphQL은 단일 엔드포인트에서 클라이언트가 필요한 데이터를 쿼리합니다. REST는 오버/언더 페칭 문제가 있고, GraphQL은 이를 해결하지만 캐싱이 어렵습니다.",
    "REST는 URL을 사용하고 GraphQL은 쿼리를 사용합니다.",
    "둘 다 API입니다.",
]


for i, ans in enumerate(answers):
    result = pointwise_judge(question, ans)
    print(f"답변 : {i+1} : {ans}")
    print(f" 점수: {result.get('score')}/5 - {result.get('reasoning')[:30]}")

답변 : 1 : REST는 리소스 단위 URL에 HTTP 메서드를 사용하고, GraphQL은 단일 엔드포인트에서 클라이언트가 필요한 데이터를 쿼리합니다. REST는 오버/언더 페칭 문제가 있고, GraphQL은 이를 해결하지만 캐싱이 어렵습니다.
 점수: 4/5 - 답변은 REST와 GraphQL의 차이점에 대해 정확하
답변 : 2 : REST는 URL을 사용하고 GraphQL은 쿼리를 사용합니다.
 점수: 2/5 - 답변은 REST API와 GraphQL의 기본적인 차이
답변 : 3 : 둘 다 API입니다.
 점수: 1/5 - 답변이 질문에 대한 구체적인 정보나 차이점을 제공하지 


## Pairwise 평가

pairwise

LLM, Lost-in-the-middle -> (A, B) -> (B, A) 등의 순으로 입력 변경해보거나
- 점수가 같을 경우
- 점수가 같은데 길이가 비슷한 경우
- 점수 차이가 비슷한데..
- 여러 사례를 테스트

### 구현: `pairwise_judge`

두 답변 A, B를 LLM에 동시에 제시 → 더 나은 답변 선택.  
JSON 응답 형식 강제로 구조화된 결과 반환.

In [139]:
def pairwise_judge(question, answer_a, answer_b):
    
    prompt = f"""두 답변을 비교하여 더 나은쪽 선택하세요
    
    질문 : {question}
    
    답변 A : {answer_a}
    
    답변 B : {answer_b}
    
    반드시 JSON으로 답하세요:
    {{"winner" : "A 또는 B 또는 Tie", "reasoning" : "선택 이유"}}"""
    
    result = llm.invoke(prompt).content
    
    print(result)
    return json.loads(result.replace('```json', '').replace('```',''))

In [140]:
question = "마이크로서비스 아키텍처란?"
answer_a = "마이크로서비스는 애플리케이션을 작은 독립 서비스로 분리하는 아키텍처 패턴입니다. 각 서비스는 독립 배포, 확장이 가능합니다."
answer_b = "여러 개의 작은 서비스로 나누는 것입니다."


In [141]:
pairwise_judge(question, answer_a, answer_b)

```json
{"winner" : "A", "reasoning" : "답변 A는 마이크로서비스 아키텍처의 정의와 특징을 상세하게 설명하고 있으며, 독립 배포와 확장 가능성 등을 포함해 더 깊이 있는 정보를 제공하고 있습니다. 반면, 답변 B는 개념을 간단히 설명할 뿐, 세부 사항이 부족합니다."}
```


{'winner': 'A',
 'reasoning': '답변 A는 마이크로서비스 아키텍처의 정의와 특징을 상세하게 설명하고 있으며, 독립 배포와 확장 가능성 등을 포함해 더 깊이 있는 정보를 제공하고 있습니다. 반면, 답변 B는 개념을 간단히 설명할 뿐, 세부 사항이 부족합니다.'}

In [142]:
pairwise_judge(question, answer_b, answer_a)

{"winner" : "B", "reasoning" : "답변 B는 마이크로서비스 아키텍처의 본질을 더 잘 설명하고 있으며, 서비스의 독립성, 배포, 확장성 등의 중요한 특징을 언급하고 있습니다. 이에 비해 답변 A는 지나치게 간단하고 구체적인 정보를 제공하지 않습니다."}


{'winner': 'B',
 'reasoning': '답변 B는 마이크로서비스 아키텍처의 본질을 더 잘 설명하고 있으며, 서비스의 독립성, 배포, 확장성 등의 중요한 특징을 언급하고 있습니다. 이에 비해 답변 A는 지나치게 간단하고 구체적인 정보를 제공하지 않습니다.'}

### Pairwise with Swap

같은 질문을 (A→B), (B→A) 순서로 두 번 평가.  
결과가 다르면 위치 편향(Position Bias) 존재를 의심.

In [143]:
def pairwise_judge_with_swap(question, answer_a, answer_b):
    
    result1 = pairwise_judge(question, answer_a, answer_b)
    
    result2 = pairwise_judge(question, answer_b, answer_a)
    
    swapped_winner = {'A':'B', 'B': 'A', 'Tie':'Tie'}.get(result2['winner'])
    
    if result1['winner'] == swapped_winner:
        final = result1['winner']
        status = '일치'
    else:
        final = 'Tie'
        status = '불일치'
        
    return {
        'final' : final,
        'status' : status
    }

In [144]:
pairwise_judge_with_swap(question, answer_a, answer_b)

{"winner" : "A", "reasoning" : "답변 A는 마이크로서비스 아키텍처의 의미와 이점인 독립적인 배포 및 확장 가능성을 설명하고 있어 더 포괄적이고 유용한 정보입니다. 반면, 답변 B는 너무 간단하여 충분한 이해를 제공하지 못합니다."}
```json
{"winner" : "B", "reasoning" : "답변 B는 마이크로서비스 아키텍처의 핵심 개념인 독립적인 서비스 분리, 배포 및 확장 가능성을 명확하게 설명하고 있어 보다 충분한 정보를 제공합니다."}
```


{'final': 'A', 'status': '일치'}

### 위치 편향 (Position Bias) 테스트

LLM은 입력에서 앞에 등장하는 텍스트에 더 주의를 기울이는 경향 (Lost-in-the-middle 현상).  
`detect_position_bias`: 순서를 바꿔 n_trials회 반복 → 위치에 따라 답이 달라지는 비율 측정.

편향률이 높을수록 → 평가가 내용보다 위치에 의존하고 있음.

In [145]:
def detect_position_bias(question, answer_a, answer_b, n_trials=3):
    
    consistent =  0
    inconsistent = 0
    
    for trial in range(n_trials):
        result = pairwise_judge_with_swap(question, answer_a, answer_b)
        if result['status'] == '일치':
            consistent +=1
        else:
            inconsistent +=1
            
    total = consistent + inconsistent
    bias_rate = inconsistent /  total
    return bias_rate

In [146]:
detect_position_bias(question, answer_a, answer_b, n_trials=3)

{
  "winner": "A",
  "reasoning": "답변 A는 마이크로서비스 아키텍처의 개념을 보다 충분히 설명하고 '독립 배포'와 '확장' 같은 중요한 특징을 포함하고 있어 이해하기 쉽고 유익합니다. 반면 답변 B는 너무 간단하여 필요한 정보를 제공하지 않습니다."
}
{"winner" : "B", "reasoning" : "답변 B는 마이크로서비스 아키텍처의 정의를 보다 구체적이고 명확하게 설명하고 있으며, 독립 배포와 확장의 가능성에 대한 중요 정보를 포함하고 있습니다. 반면, 답변 A는 너무 간단하여 마이크로서비스의 본질을 충분히 전달하지 못합니다."}
```json
{"winner" : "A", "reasoning" : "답변 A는 마이크로서비스 아키텍처의 개념을 좀 더 상세하게 설명하고 있으며, 독립 배포와 확장 가능성이라는 중요한 특징을 언급하고 있습니다. 반면 답변 B는 단순하고 다소 모호하게 표현되어 있어, 정보의 깊이나 명확성이 부족합니다."}
```
{"winner" : "B", "reasoning" : "답변 B는 마이크로서비스 아키텍처에 대한 보다 정확하고 구체적인 설명을 제공하며, 독립적인 배포와 확장 가능성 같은 중요한 특성을 강조합니다. 반면, 답변 A는 매우 간단한 정의에 불과하여 정보가 부족합니다."}
```json
{"winner" : "A", "reasoning" : "답변 A는 마이크로서비스 아키텍처의 정의를 명확하게 설명하고 있으며, 독립 배포 및 확장 가능성과 같은 중요한 특징을 포함하고 있습니다. 반면 답변 B는 너무 간단하여 개념을 충분히 전달하지 못합니다."}
```
```json
{"winner" : "B", "reasoning" : "답변 B는 마이크로서비스 아키텍처의 본질을 잘 설명하고 있으며, 독립적인 배포와 확장 가능성이라는 중요한 특징을 언급하고 있습니다. 반면에 답변 A는 너무 간단하게 설명되어 구체적인 내용을 담고 있지 않습니다."}
```


0.0

### Reference-Based 평가

정답(Reference)이 있는 경우 활용. LLM이 정답과 학생 답변을 비교해 평가:

1. 정답의 핵심 포인트 포함 여부
2. 사실적 오류 유무
3. 정답에 없는 올바른 추가 정보 여부

In [147]:
def reference_based_judge(question, reference, answer):
    prompt = f"""당신은 AI 교육 전문가입니다.
    정답을 기준으로 학생의 답변을 평가하세요.
    
    질문 : {question}
    정답 : {reference}
    학생 답변 : {answer}
    
    평가 기준:
    - 정답의 핵심 포인트를 얼마나 포함하는가
    - 사실적으로 틀린 내용이 있는가
    - 정답에 없는 올바른 추가 정보가 있는가
    
    JSON으로 답하세요:
    {{"score": 1-5, "covered_points" : ["포함된 핵심 포인트들"], "missing_points":["누락된 포인트들"], "errors" : ["틀린 내용"]}}"""
    
    result = llm.invoke(prompt).content
    cleaned = result.replace('```json', '').replace('```', '')
    return json.loads(cleaned)

In [148]:
question = "TCP와 UDP의 차이점은?"
reference = "TCP는 연결지향적이고 신뢰성 있는 전송을 보장하며, UDP는 비연결지향적이고 빠르지만 신뢰성을 보장하지 않습니다."
answer = "TCP는 연결을 먼저 수립하고 데이터 전송의 순서와 무결성을 보장합니다. 반면 UDP는 연결 없이 바로 전송하여 빠르지만 패킷 손실이 가능합니다."


In [149]:
result = reference_based_judge(question, reference, answer)

In [150]:
result

{'score': 5,
 'covered_points': ['TCP는 연결을 먼저 수립한다',
  '데이터 전송의 순서와 무결성을 보장한다',
  'UDP는 연결 없이 바로 전송한다',
  'UDP는 빠르지만 패킷 손실이 가능하다'],
 'missing_points': ['TCP는 신뢰성 있는 전송을 보장한다', 'UDP는 신뢰성을 보장하지 않는다'],
 'errors': []}

## 부록: RAG 개념 정리

Naive RAG -> 문서를 쪼갠 다음에, 질문에 어울리는 파트들을 유사도? 등의 기준을 통해 가져옴

Advanced RAG , Reranking

Multi-hop : hop은 뛰어다니는거지..
- A 고객이 b 제품을 구매했다.
- b 제품은 1번 공장에서 만들어졌다.
- 1번 공장의 책임자는 abc이다.

B 고객이 A 고객과 같은 물건을 샀어요, 공장 책임자가 누군가요? --> Multi-hop 질문의 예시

GraphRAG -> Multi-hop 예문과 같은 것들을 전부다 그래프 형태로 바꿔줌
- A -> b
- b -> 1
- 1 -> abc

지금 하고있는 메트릭을 하다보면 GraphRAG에서 답변 등이 어떻게 이어지는지 정량평가 가능